# 7.1 A First Prediction Model

**Machine learning series, class 1.** Can we tell, on the day someone joins *All of Us*, who is likely to be diagnosed with type 2 diabetes in the next **2 years**? In this class we build that prediction from what's known at enrollment: age, sex, BMI, blood pressure, and a few survey answers. We fit a **logistic regression**, test it on people it has never seen, and then try **LASSO**, a method that chooses predictors for us.

New to machine learning? Read [Machine learning for beginners](https://cwml.github.io/All-of-Us-Training-Resources/pages/courses/ml/ml-for-beginners.html) first: it explains every word in **bold** below.

### By the end of this notebook you'll be able to

1. Turn a research question into a **prediction** question: who, from what, predicting what, by when
2. Build an outcome that only uses the **future**, and predictors that only use the **past** (no **leakage**)
3. Split the data into **training** and **test** sets, and say why
4. Fit a **logistic regression** with **tidymodels**, and measure it with the **AUC**
5. Check **calibration**: do predicted risks match what happened?
6. Tune a **LASSO** with **cross-validation**, and compare it with the simple model

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | From what's known at enrollment, how well can we predict a **new** type 2 diabetes diagnosis within 2 years? |
| 🧪 **The analysis** | Adults without diabetes at enrollment, with BMI and blood pressure measured then; a 75/25 train/test split; logistic regression (age and sex, + BMI, + everything), then LASSO tuned by 5-fold cross-validation |
| 🎯 **Why** | Logistic regression is the standard first model for a yes/no outcome. Testing on held-out people shows how it would do on **new** people |
| ✅ **What we conclude** | How well enrollment data separate people who will and won't be diagnosed (AUC), whether the risks are right (calibration), and how much each kind of data adds |
| ⚠️ **What it can't tell us** | Cause: a predictor can predict without causing anything. And whether the model works outside *All of Us*, or for groups it saw few of |
| 🔗 **Connections** | The cohort query from **1.1**; plausible values from **2.1**; logistic regression from **3.2**; survey answers from **5.1–5.3**; "measure first, then follow" from **6.2**; time zero from the ML data check |

| | |
|---|---|
| 📂 **Uses** | the Controlled Tier dataset: the shared cohort query, survey answers, BMI and blood pressure |
| 💾 **Saves** | `ml_t2d_2yr.csv` (one row per person in the analysis), for **7.2** |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: a prediction question | 15 |
| 2. Setup and the cohort | 5 |
| 3. What's known at enrollment, in one query | 10 |
| 4. Who's in the analysis, and the outcome | 10 |
| 5. Training and test sets | 5 |
| 6. Logistic regression, and the AUC | 15 |
| 7. ROC curve and calibration | 15 |
| 8. LASSO with cross-validation | 15 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 95 minutes |
| ⏱️ **Run time** | about **10 minutes**, plus up to **15 minutes the first time** to install the modelling packages (tidymodels) |
| 💰 **Data cost** | about **\$1 to \$2**: the cohort query, and one query for everyone's enrollment data, priced before it runs |
| 💰 **App cost** | about **\$0.27 an hour** while the app is running. The app's disk costs about **\$0.70 a day**, even while the app is stopped |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### Explaining vs predicting

In **3.2** and **5.3** we asked: *is income linked to type 2 diabetes, after accounting for age and sex?* That's **explanation**: we cared about one odds ratio, and about confounding.

Here we ask: *for a new person, what's their chance of being diagnosed?* That's **prediction**: we care about how **accurate** the predicted risks are, for people the model hasn't seen. A predictor can be useful without causing anything.

| | Explaining (3.2, 5.3) | Predicting (this series) |
|---|---|---|
| The goal | one effect, with a confidence interval | accurate risks for new people |
| Judged by | is the odds ratio unbiased? | **AUC** and **calibration** on a **test set** |
| Confounders | must be adjusted for | anything known in time can help |
| The worry | confounding | **overfitting** and **leakage** |

### The prediction question

```
                enrollment (time zero)                       2 years later
                        |──────────── follow-up ──────────────|
  predictors: age, sex, BMI and blood pressure     outcome: a NEW type 2 diabetes
  measured around this day; survey answers         diagnosis in these 2 years? yes / no
  given by 30 days after it
  excluded: type 2 diabetes on or before this day
```

| | Our choice | Why |
|---|---|---|
| **Who** | adults with no diabetes at enrollment | you can't predict a diagnosis that has already happened. In the ML data check, **77%** of cases were diagnosed **before** they enrolled |
| **From what** | data from around enrollment only | a model used on the day someone joins can only use what's known that day |
| **Predicting what** | a new type 2 diabetes code within 2 years | the shared case definition (**1.1**), dated |
| **Who counts** | people seen in health records for 2 years, or diagnosed sooner | someone whose records stop after 6 months might have been diagnosed later without us seeing it |

### Leakage: the most common way to fool yourself

**Leakage** is when a predictor contains information about the outcome that you wouldn't have at prediction time. Examples here: an HbA1c test (it's how diabetes is **diagnosed**), a metformin prescription (a **treatment**), or a survey answered **after** the diagnosis. A leaky model looks brilliant on paper and fails in practice. We avoid it by **dating** every predictor.

## Part 2. Setup and the cohort

`install_if_missing()` (in `aou_setup.R`) installs the modelling packages the first time; after that it's skipped. **tidymodels** is a set of packages that share one way of working:

| Package | Does |
|---|---|
| `rsample` | training/test splits, cross-validation folds |
| `recipes` | preparing predictors (dummy variables, splines, scaling) |
| `parsnip` | one way to specify many models (here `logistic_reg()`) |
| `workflows` | a recipe and a model, together |
| `tune`, `dials` | trying settings, e.g. the LASSO penalty |
| `yardstick` | measuring performance (AUC) |

`HORIZON_YEARS` is the prediction window. Leave it at 2 for now: you'll change it in **Your turn**.

💰 The cohort query: cents to under \$1.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

ML_PACKAGES <- c("rsample", "recipes", "parsnip", "workflows", "tune", "dials", "yardstick", "glmnet", "splines2")
install_if_missing(ML_PACKAGES)   # first time only: up to 15 minutes
suppressPackageStartupMessages(for (pkg in c(ML_PACKAGES, "broom")) library(pkg, character.only = TRUE))

HORIZON_YEARS <- 2                      # predict a diagnosis within this many years
end_of_data   <- as.Date("2025-01-01")  # the CDR v9 data cutoff

cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id),
         t2d_date  = as.Date(first_t2d_date),
         last_ehr  = as.Date(last_ehr_date),
         death     = as.Date(death_date))

cohort %>% count(cohort) %>% hide_small_counts()

**🔍 Read the output:** cases, controls, and excluded people (type 1 diabetes, or unclear), as in **1.1**: over 400,000 adults with EHR data. *Installing…* messages the first time are normal.

## Part 3. What's known at enrollment, in one query

### First, check the concept IDs

Never guess a concept ID: look it up. The survey questions are the ones we used in **5.1** and **5.3**; education was found by its **concept code** in **5.3**, so we look up its ID the same way.

In [ ]:
QUESTION_IDS <- c(general_health = 1585711, income = 1585375, smoked_100 = 1585857, smoke_frequency = 1585860)

concepts <- run_query(str_glue("
  SELECT concept_id, concept_code, concept_name
  FROM `{CURRENT_DATASET}.concept`
  WHERE concept_id IN ({str_c(QUESTION_IDS, collapse = ', ')}, 3038553, 3004249, 4152194)
     OR concept_code = 'EducationLevel_HighestGrade'"))

concepts

EDUCATION_ID <- concepts$concept_id[concepts$concept_code == "EducationLevel_HighestGrade"]
if (length(EDUCATION_ID) != 1) stop("Expected one education question, found ", length(EDUCATION_ID), ": check the table above")
QUESTION_IDS <- c(QUESTION_IDS, education = EDUCATION_ID)
QUESTION_IDS

**🔍 Read the output:** one row per concept, with its name. Check they mean what we think: BMI, two systolic blood pressure concepts (**2.1**), general health, income, two smoking questions, and education. Then the five question IDs we'll use. In our test run, education was **1585940** (`EducationLevel_HighestGrade`).

### The enrollment query

One query, one row per person who answered a survey. It's a chain of named steps (**1.1**):

| Step | Does |
|---|---|
| `enroll` | enrollment = the date of each person's **first survey** (as in the ML data check) |
| `measures` | BMI (12 to 80) and systolic blood pressure (60 to 260 mm Hg), the plausible ranges from **2.1**, measured from 1 year before to 30 days after enrollment |
| `closest` | for each person, the BMI and blood pressure **closest** to enrollment |
| `answers` | answers to our five questions, given **by 30 days after** enrollment: later answers could be leakage |
| `first_answers` | each person's **first** answer to each question |

`ARRAY_AGG(x IGNORE NULLS ORDER BY y LIMIT 1)[SAFE_OFFSET(0)]` means "the first `x`, sorted by `y`": BigQuery's way of picking one value per person.

💰 `run_if_cheap()` prices it first. In the ML data check, a similar query cost under \$2.

In [ ]:
features_sql <- str_glue("
WITH enroll AS (
  SELECT person_id, MIN(DATE(survey_datetime)) AS enroll_date
  FROM `{CURRENT_DATASET}.ds_survey`
  GROUP BY person_id
),
measures AS (
  SELECT m.person_id,
         IF(m.measurement_concept_id = 3038553, 'bmi', 'sbp') AS measure,
         m.value_as_number,
         ABS(DATE_DIFF(m.measurement_date, e.enroll_date, DAY)) AS days_from_enroll
  FROM `{CURRENT_DATASET}.measurement` AS m
  JOIN enroll AS e ON m.person_id = e.person_id
  WHERE ((m.measurement_concept_id = 3038553 AND m.value_as_number BETWEEN 12 AND 80)                -- BMI
      OR (m.measurement_concept_id IN (3004249, 4152194) AND m.value_as_number BETWEEN 60 AND 260))  -- systolic BP
    AND m.measurement_date BETWEEN DATE_SUB(e.enroll_date, INTERVAL 1 YEAR)
                               AND DATE_ADD(e.enroll_date, INTERVAL 30 DAY)
),
closest AS (
  SELECT person_id,
         ARRAY_AGG(IF(measure = 'bmi', value_as_number, NULL) IGNORE NULLS
                   ORDER BY days_from_enroll LIMIT 1)[SAFE_OFFSET(0)] AS bmi,
         ARRAY_AGG(IF(measure = 'sbp', value_as_number, NULL) IGNORE NULLS
                   ORDER BY days_from_enroll LIMIT 1)[SAFE_OFFSET(0)] AS sbp
  FROM measures
  GROUP BY person_id
),
answers AS (
  SELECT s.person_id, s.question_concept_id, s.answer, s.survey_datetime
  FROM `{CURRENT_DATASET}.ds_survey` AS s
  JOIN enroll AS e ON s.person_id = e.person_id
  WHERE s.question_concept_id IN ({str_c(QUESTION_IDS, collapse = ', ')})
    AND DATE(s.survey_datetime) <= DATE_ADD(e.enroll_date, INTERVAL 30 DAY)
),
first_answers AS (
  SELECT person_id,
    {str_c('ARRAY_AGG(IF(question_concept_id = ', QUESTION_IDS, ', answer, NULL) IGNORE NULLS ',
           'ORDER BY survey_datetime LIMIT 1)[SAFE_OFFSET(0)] AS ', names(QUESTION_IDS), collapse = ',\n    ')}
  FROM answers
  GROUP BY person_id
)
SELECT e.person_id, e.enroll_date, c.bmi, c.sbp,
       a.general_health, a.income, a.education, a.smoked_100, a.smoke_frequency
FROM enroll AS e
LEFT JOIN closest       AS c ON e.person_id = c.person_id
LEFT JOIN first_answers AS a ON e.person_id = a.person_id")

features_raw <- run_if_cheap(features_sql, max_dollars = 3) %>%
  mutate(person_id = as.character(person_id))

last_query_cost()

features_raw %>%
  summarise(people             = n(),
            pct_bmi            = round(100 * mean(!is.na(bmi)), 1),
            pct_sbp            = round(100 * mean(!is.na(sbp)), 1),
            pct_general_health = round(100 * mean(!is.na(general_health)), 1),
            pct_income         = round(100 * mean(!is.na(income)), 1),
            pct_education      = round(100 * mean(!is.na(education)), 1),
            pct_smoking        = round(100 * mean(!is.na(smoked_100)), 1))

**🔍 Read the output:** the cost; then everyone who answered a survey, and the percentage with each predictor at enrollment.

In our test run the query read **31.8 GB (about \$0.20)**, for **747,029** people: about 70% had BMI and blood pressure at enrollment, 82% a general health answer, 93% income and education, and 81% the smoking question. Rerunning the same query within a day costs **\$0**: BigQuery reuses the saved result (`last_query_cost()` shows 0.00 GB).

*"Long rows: downloading again in smaller pages"* may appear: 747,029 rows of survey text are too much for one download page, so `run_query()` downloads them again in smaller pages, without rerunning the query. The survey questions come from *The Basics*, *Overall Health*, and *Lifestyle*, the surveys people answer when they join, so most people have them.

### From answers to predictors

The same recoding as in **5.1** and **5.3**. A missing or skipped answer becomes its own group, **Missing**: in prediction, *not answering* can itself carry information, and it keeps everyone in the model.

In [ ]:
# From 5.1: drop the question's short name ("General Health: Excellent" becomes "Excellent"),
# but leave non-answers as they are ("PMI: Skip")
answer_text <- function(answer) {
  if_else(str_starts(answer, "PMI:"), answer, str_remove(answer, "^.*: "))
}

features <- features_raw %>%
  mutate(across(c(general_health, income, education, smoked_100, smoke_frequency), answer_text)) %>%
  mutate(
    general_health = if_else(general_health %in% c("Excellent", "Very Good", "Good", "Fair", "Poor"),
                             general_health, NA_character_),
    income = case_when(                                                    # 5.3, in US dollars a year
      income %in% c("less 10k", "10k 25k")                 ~ "Under 25k",     # no $ signs: JupyterLab shows
      income %in% c("25k 35k", "35k 50k")                  ~ "25k to 50k",    # text between two $ as a formula
      income %in% c("50k 75k", "75k 100k")                 ~ "50k to 100k",
      income %in% c("100k 150k", "150k 200k", "more 200k") ~ "100k or more",
      income == "PMI: Prefer Not To Answer"                ~ "Prefer not to answer"),
    education = case_when(                                                 # 5.3
      is.na(education) | str_starts(education, "PMI:")                                  ~ NA_character_,
      str_detect(education, regex("never|one through|five through|nine through", ignore_case = TRUE)) ~ "Less than high school",
      str_detect(education, regex("twelve|GED", ignore_case = TRUE))                    ~ "High school or GED",
      str_detect(education, regex("college one|one to three", ignore_case = TRUE))      ~ "Some college",
      str_detect(education, regex("college graduate|advanced", ignore_case = TRUE))     ~ "College degree",
      TRUE                                                                              ~ "CHECK"),
    smoking_status = case_when(                                            # 5.1
      str_detect(smoked_100, regex("^no$", ignore_case = TRUE))                     ~ "Never",
      str_detect(smoke_frequency, regex("every day|some days", ignore_case = TRUE)) ~ "Current",
      str_detect(smoked_100, regex("^yes$", ignore_case = TRUE)) &
        str_detect(smoke_frequency, regex("not at all", ignore_case = TRUE))        ~ "Former")) %>%
  mutate(across(c(general_health, income, education, smoking_status), ~ replace_na(.x, "Missing")),
         general_health = factor(general_health, levels = c("Excellent", "Very Good", "Good", "Fair", "Poor", "Missing")),
         income         = factor(income, levels = c("100k or more", "50k to 100k", "25k to 50k", "Under 25k",
                                                    "Prefer not to answer", "Missing")),
         education      = factor(education, levels = c("College degree", "Some college", "High school or GED",
                                                       "Less than high school", "CHECK", "Missing")),
         smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current", "Missing"))) %>%
  select(person_id, enroll_date, bmi, sbp, general_health, income, education, smoking_status)

features %>%
  select(general_health, income, education, smoking_status) %>%
  pivot_longer(everything(), names_to = "predictor", values_to = "group") %>%
  count(predictor, group) %>%
  hide_small_counts()

**🔍 Read the output:** how many people are in each group of each predictor. Check:

- `education`: **CHECK** means an answer the recoding didn't recognise. If you see it, look at those answers (`features_raw %>% count(education)`) and fix the `case_when()`. Any left are treated as **Missing** below.
- `Missing` is larger for smoking than for the others: the *Lifestyle* survey is the last of the three enrollment surveys, and some people stop before it.

## Part 4. Who's in the analysis, and the outcome

We join the predictors to the cohort, then apply the rules from Part 1 one at a time, **counting** how many people each removes (a **flow table**, as in **6.2**).

In [ ]:
people <- cohort %>%
  select(person_id, cohort, date_of_birth, sex_at_birth, t2d_date, last_ehr, death) %>%
  inner_join(features, by = "person_id") %>%
  mutate(age           = as.numeric(enroll_date - as.Date(date_of_birth)) / 365.25,
         horizon_end   = enroll_date + round(365.25 * HORIZON_YEARS),
         followup_end  = pmin(last_ehr, death, end_of_data, na.rm = TRUE),
         t2d_by_enroll = cohort == "case" & t2d_date <= enroll_date,
         new_t2d       = cohort == "case" & t2d_date > enroll_date & t2d_date <= horizon_end,
         education     = fct_collapse(education, Missing = c("CHECK", "Missing")))

rules <- list(
  rep(TRUE, nrow(people)),
  people$cohort != "excluded",
  !people$t2d_by_enroll,
  people$age >= 18 & people$sex_at_birth %in% c("Female", "Male"),
  !is.na(people$bmi) & !is.na(people$sbp),
  people$new_t2d | people$followup_end >= people$horizon_end)
names(rules) <- c("in the cohort query, with an enrollment survey",
                  "not 'excluded' (type 1 or unclear diabetes)",
                  "no type 2 diabetes by enrollment",
                  "18 or older at enrollment; sex at birth Female or Male",
                  "BMI and blood pressure at enrollment",
                  str_glue("health records for {HORIZON_YEARS} years after enrollment, or diagnosed sooner"))
keep <- accumulate(rules, `&`)       # each rule, and every rule before it

tibble(rule = names(rules), people = map_int(keep, sum)) %>%
  mutate(removed = lag(people) - people,
         across(c(people, removed), ~ if_else(removed %in% 1:20, NA_integer_, .x)))   # hide small differences

**🔍 Read the output:** how many people are left after each rule, and how many each removed. In our test run:

| Rule | People | Removed |
|---|---|---|
| in the cohort query, with an enrollment survey | 426,596 | |
| not "excluded" | 412,831 | 13,765 |
| no type 2 diabetes by enrollment | 350,404 | 62,427 |
| 18 or older; sex at birth Female or Male | 346,925 | 3,479 |
| BMI and blood pressure at enrollment | 295,908 | 51,017 |
| health records for 2 years, or diagnosed sooner | **152,586** | 143,322 |

- **No type 2 diabetes by enrollment** removes the most cases: most were diagnosed before they joined.
- **Health records for 2 years** removes the most people: those who enrolled recently (after early 2023, there are not yet 2 years of data) and people whose records stop. This is a real limitation: if people who stop visiting differ from those who keep visiting, our sample is less typical. A **survival** model (**3.4**, **6.2**) can use everyone's follow-up, however short; we keep a yes/no outcome to start simple.
- The sex rule keeps the model to groups large enough to estimate. A model used in practice would need a plan for everyone.

In [ ]:
PREDICTORS <- c("age", "sex_at_birth", "bmi", "sbp", "general_health", "income", "education", "smoking_status")

ml_data <- people[keep[[length(keep)]], ] %>%
  mutate(t2d_future   = factor(if_else(new_t2d, "yes", "no"), levels = c("no", "yes")),
         sex_at_birth = factor(sex_at_birth)) %>%
  select(person_id, enroll_date, t2d_future, all_of(PREDICTORS)) %>%
  mutate(across(where(is.factor) & !t2d_future, fct_drop))   # drop groups with nobody left in them

ml_data %>%
  summarise(people        = n(),
            new_t2d       = sum(t2d_future == "yes"),
            pct_new_t2d   = round(100 * mean(t2d_future == "yes"), 2),
            enrolled_from = year(min(enroll_date)),
            enrolled_to   = year(max(enroll_date)))

**🔍 Read the output:** the people in the analysis, how many were diagnosed with type 2 diabetes within 2 years, as a percentage, and the years they enrolled. In our test run: **152,586** people, **8,487** new diagnoses (**5.6%**), enrolled from 2017 to 2024.

The percentage is small: a **rare outcome**. That matters for every number that follows. A model that says "no" to everyone is right for almost everyone, and useless. This is why we don't judge models by **accuracy**.

The outcome is a factor with levels `no`, `yes`: `no` first, as the **reference** (as in **3.2**). We tell the AUC function that `yes` (the **second** level) is the event.

## Part 5. Training and test sets

We hide a quarter of the people in a **test set**, and don't look at it until the end. Every model is fitted on the **training set** only. The test set then plays the part of **new people**: the AUC on it is an honest estimate of how the model would do on people it has never seen.

`strata = t2d_future` makes both sets have the same share of new cases. `set.seed()` makes the random split the same every time you run it (and the same for everyone in the class).

In [ ]:
model_data <- ml_data %>% select(t2d_future, all_of(PREDICTORS))

set.seed(2026)
split <- initial_split(model_data, prop = 0.75, strata = t2d_future)
train <- training(split)
test  <- testing(split)

bind_rows(training = count(train, t2d_future), test = count(test, t2d_future), .id = "set") %>%
  group_by(set) %>%
  mutate(percent = round(100 * n / sum(n), 2)) %>%
  ungroup()

**🔍 Read the output:** about 75% of people in training, 25% in test, with the same percentage of new cases in each.

## Part 6. Logistic regression, and the AUC

We fit three logistic regressions, each adding more of what's known at enrollment. `fit_logistic()` takes a list of predictors and fits on the **training** set. `test_auc()` predicts for the **test** set and measures the **AUC**:

| AUC | Means |
|---|---|
| 0.5 | no better than a coin toss |
| 0.7 | acceptable; typical of models from routine data |
| 0.8 | good |
| 1.0 | perfect: suspect **leakage** |

The AUC is the chance that a randomly chosen person **who was** diagnosed got a higher predicted risk than a randomly chosen person **who wasn't**.

In [ ]:
fit_logistic <- function(predictors) {
  workflow() %>%
    add_formula(reformulate(predictors, response = "t2d_future")) %>%
    add_model(logistic_reg()) %>%
    fit(data = train)
}

test_auc <- function(fit, model) {
  augment(fit, new_data = test) %>%                                   # adds .pred_yes: the predicted risk
    roc_auc(truth = t2d_future, .pred_yes, event_level = "second") %>%
    transmute(model, test_auc = round(.estimate, 3))
}

MODELS <- list("1. Age and sex"           = c("age", "sex_at_birth"),
               "2. + BMI"                 = c("age", "sex_at_birth", "bmi"),
               "3. All enrollment data"   = PREDICTORS)

fits <- map(MODELS, fit_logistic)
auc_table <- imap(fits, test_auc) %>% list_rbind()
auc_table

**🔍 Read the output:** the test-set AUC for each model.

In our test run:

| Model | Test AUC |
|---|---|
| 1. Age and sex | 0.577 |
| 2. + BMI | 0.709 |
| 3. All enrollment data | **0.743** |

- **Age and sex** alone do only a little better than a coin toss. Everyone here was **free** of diabetes at enrollment, at any age, so age separates them less than you might expect.
- **BMI** adds the most: it's the strongest single risk factor for type 2 diabetes.
- **Everything else** (blood pressure, general health, income, education, smoking) adds a useful amount more. A difference of under about **0.01** is within the noise of a test set this size.
- **0.74** is typical of models built from routine data such as these.

### The odds ratios: explanation again?

The model is a logistic regression, so it has odds ratios, as in **3.2**. They show **what the model uses**. Read them with care: they aren't causal effects, and the predictors are adjusted for each other.

In [ ]:
fits[["3. All enrollment data"]] %>%
  extract_fit_parsnip() %>%
  tidy() %>%
  filter(term != "(Intercept)") %>%
  transmute(term,
            OR      = round(exp(estimate), 2),
            CI_low  = round(exp(estimate - 1.96 * std.error), 2),
            CI_high = round(exp(estimate + 1.96 * std.error), 2))

**🔍 Read the output:** one row per predictor (or group, compared with the **first** group). `age` is per year, `bmi` per unit (kg/m²), `sbp` per mm Hg: small per unit, large over 10 or 20 units. In our test run: OR **1.07** per BMI unit (about **1.40** per 5 units), **1.43** for men, **1.02** per year of age; and a steep gradient by general health, from **1.27** (*Very Good*) to **4.74** (*Poor*), compared with *Excellent*. People who already feel unwell may have **undiagnosed** diabetes. That helps **prediction**, but it's no cause. Income and education have smaller gradients (up to **1.39** and **1.56**). Smoking adds almost nothing once the rest is known.

## Part 7. ROC curve and calibration

### The ROC curve

The AUC is the **area under** the ROC curve. Each point is one **cut-off**: everyone with a predicted risk above it is "flagged". **Sensitivity** is the share of future cases flagged; **1 − specificity** is the share of everyone else flagged (false alarms). A useless model follows the dashed diagonal.

`roc_points()` uses 50 cut-offs, one every 2% of predicted risks, so every point summarises **hundreds** of people. (yardstick's `roc_curve()` uses every person's own risk as a cut-off: fine in the Workbench, but a plot of it shouldn't leave it.)

In [ ]:
roc_points <- function(fit, model) {
  pred <- augment(fit, new_data = test)
  cuts <- c(-Inf, quantile(pred$.pred_yes, probs = seq(0.02, 0.98, by = 0.02)), Inf)
  tibble(model,
         sensitivity = map_dbl(cuts, ~ mean(pred$.pred_yes[pred$t2d_future == "yes"] >= .x)),
         specificity = map_dbl(cuts, ~ mean(pred$.pred_yes[pred$t2d_future == "no"] < .x)))
}

imap(fits, roc_points) %>%
  list_rbind() %>%
  ggplot(aes(x = 1 - specificity, y = sensitivity, colour = model)) +
  geom_abline(linetype = "dashed", colour = "grey50") +
  geom_path(linewidth = 1) +
  coord_equal() +
  labs(x = "1 - specificity (false alarms)", y = "Sensitivity (future cases flagged)", colour = NULL,
       title = str_glue("Predicting type 2 diabetes within {HORIZON_YEARS} years"),
       subtitle = "Test set; each curve is one model") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom", legend.direction = "vertical")

**🔍 Read the output:** the higher a curve bows towards the top left, the better. In our test run the age-and-sex curve stayed close to the diagonal; adding BMI lifted it most; model 3 sat a little above model 2 all the way along.

### Calibration: are the risks right?

The AUC only checks the **order** of the risks. A model can rank people well and still say *10%* when the truth is *2%*. **Calibration** checks the numbers: we put the test set into 10 equal groups by predicted risk (**deciles**), and compare the average predicted risk with the share actually diagnosed.

In [ ]:
test_pred <- augment(fits[["3. All enrollment data"]], new_data = test)

calibration <- test_pred %>%
  mutate(risk_group = ntile(.pred_yes, 10)) %>%
  group_by(risk_group) %>%
  summarise(people = n(), predicted_pct = round(100 * mean(.pred_yes), 2), n = sum(t2d_future == "yes")) %>%
  hide_small_counts() %>%
  # if only one group is hidden, it could be worked out from the total: hide the next smallest too
  mutate(n = if (sum(is.na(n)) == 1) replace(n, which.min(n), NA) else n,
         observed_pct = round(100 * n / people, 2)) %>%
  rename(new_cases = n)

calibration

calibration %>%
  filter(!is.na(observed_pct)) %>%
  ggplot(aes(x = predicted_pct, y = observed_pct)) +
  geom_abline(linetype = "dashed", colour = "grey50") +
  geom_line(colour = "steelblue") +
  geom_point(size = 3, colour = "steelblue") +
  labs(x = "Predicted risk (%)", y = "Diagnosed within the window (%)",
       title = "Calibration: model 3, test set", subtitle = "One point per tenth of predicted risk") +
  theme_minimal(base_size = 14)

**🔍 Read the output:** one row per tenth of predicted risk, lowest first. Points on the dashed line mean the predicted risks are **right**. Logistic regression is usually well calibrated on the people it came from. Risk group 10 should have many times the rate of group 1.

In our test run the predicted risks rose from about **0.9%** in the lowest tenth to **18%** in the highest, and the observed rates followed them closely: from **1.7%** (group 3) to **16.4%** (group 10). Only the top group was a little **over**-predicted (18.2% predicted, 16.4% observed). Groups 1 and 2 had 1 to 20 new cases, so they're hidden.

### What would a screening rule catch?

Suppose we offered a diabetes test to the **10% with the highest predicted risk**. How many future cases would that reach?

In [ ]:
test_pred %>%
  mutate(top_10_pct = .pred_yes >= quantile(.pred_yes, 0.9)) %>%
  summarise(pct_of_future_cases_in_top_10 = round(100 * mean(top_10_pct[t2d_future == "yes"]), 1),
            pct_diagnosed_top_10          = round(100 * mean(t2d_future[top_10_pct] == "yes"), 1),
            pct_diagnosed_everyone_else   = round(100 * mean(t2d_future[!top_10_pct] == "yes"), 1))

**🔍 Read the output:**

- `pct_of_future_cases_in_top_10`: the share of everyone diagnosed in the window who was in the top 10% (the **sensitivity** of this rule).
- `pct_diagnosed_top_10`: how many of those flagged were diagnosed (the **positive predictive value**).
- `pct_diagnosed_everyone_else`: the rate in the other 90%.

In our test run: the top 10% held **30%** of the future cases; **16.4%** of them were diagnosed within 2 years, compared with **4.2%** of everyone else, about 4 times the rate.

Even a good model flags many people who won't be diagnosed, and misses some who will. Whether that's worth it depends on the cost of the test and the harm of a missed case: a clinical question, not a statistical one.

## Part 8. LASSO with cross-validation

Model 3 treats age, BMI, and blood pressure as **straight lines** on the log-odds scale. Risk might rise faster at high BMI. We can let the model **bend**, with **splines** (several columns per number), and let it use **interactions** with sex. That's many more columns, and a risk of **overfitting**: fitting the training set's noise.

**LASSO** handles this. It fits a logistic regression with a **penalty** that shrinks coefficients towards zero, and sets the least useful ones **exactly to zero**: it **chooses** predictors. The penalty's size is a **tuning parameter**: too small, and LASSO overfits; too large, and it drops useful predictors.

We choose it with **5-fold cross-validation**, using the **training set only**:

```
training set, cut into 5 folds:   [1][2][3][4][5]
round 1: fit on 2-5, score on 1
round 2: fit on 1,3-5, score on 2      ... and so on: every person is scored once,
                                           by a model that didn't see them
```

The test set stays hidden: if we used it to choose the penalty, its AUC would no longer be honest.

In [ ]:
lasso_recipe <- recipe(t2d_future ~ ., data = train) %>%
  step_spline_natural(age, bmi, sbp, deg_free = 4) %>%                  # let the numbers bend (4 columns each)
  step_dummy(all_nominal_predictors()) %>%                               # groups to 0/1 columns
  step_interact(terms = ~ starts_with("sex_at_birth"):matches("^(age|bmi|sbp)_")) %>%   # different curves by sex
  step_normalize(all_numeric_predictors())                               # same scale, so the penalty is fair

lasso_workflow <- workflow() %>%
  add_recipe(lasso_recipe) %>%
  add_model(logistic_reg(penalty = tune(), mixture = 1) %>% set_engine("glmnet"))   # mixture = 1: LASSO

set.seed(2026)
folds <- vfold_cv(train, v = 5, strata = t2d_future)

lasso_tuning <- tune_grid(lasso_workflow,
                          resamples = folds,
                          grid      = grid_regular(penalty(range = c(-5, -1)), levels = 20),
                          metrics   = metric_set(roc_auc))

autoplot(lasso_tuning) + theme_minimal(base_size = 14)

**🔍 Read the output:** the cross-validated AUC (y) for 20 penalty sizes (x, on a log scale). It's flat for small penalties, then falls as the penalty removes useful predictors. At the largest penalties every coefficient is zero, and the AUC drops to **0.5**: a model that gives everyone the same risk can't rank anyone.

We pick the **largest** penalty whose AUC is within one standard error of the best: the **simplest** model that's as good as the best (the "one-standard-error rule"). Then we refit on the whole training set, and, for the first and only time, score it on the test set.

In [ ]:
chosen <- select_by_one_std_err(lasso_tuning, desc(penalty), metric = "roc_auc")

lasso_fit <- lasso_workflow %>%
  finalize_workflow(chosen) %>%
  fit(data = train)

bind_rows(auc_table, test_auc(lasso_fit, "4. LASSO: splines and interactions"))

lasso_coefs <- lasso_fit %>% extract_fit_parsnip() %>% tidy() %>% filter(term != "(Intercept)")

c(columns = nrow(lasso_coefs), kept_by_lasso = sum(lasso_coefs$estimate != 0))

lasso_coefs %>%
  filter(estimate != 0) %>%
  arrange(desc(abs(estimate))) %>%
  transmute(term, coefficient = round(estimate, 3)) %>%
  head(10)

**🔍 Read the output:**

1. The AUC table again, with LASSO added. In our test run: **0.748**, vs **0.743** for model 3. With a few strong predictors, LASSO scores about the **same** as plain logistic regression: there's little to gain from bending lines when BMI and age carry most of the signal. LASSO pays off when there are **many** candidate predictors and few cases: **7.3**, with thousands of proteins.
2. How many columns the recipe made, and how many LASSO **kept**.
3. The 10 largest kept coefficients. In our test run, the BMI and age spline pieces and general health (*Fair*, *Poor*, *Good*) led the list. The columns are **standardised**, so sizes can be compared: a coefficient of 0.3 means the log-odds rise by 0.3 per standard deviation. `age_1` to `age_4` are the four **spline** pieces of age: read them together, as one bent line, not one by one. `sex_at_birth_Male_x_bmi_1` is an **interaction**: how the BMI curve differs for men.

## Your turn

1. **A longer window.** Change `HORIZON_YEARS <- 2` to `3` in the setup cell, and run Parts 4 to 7 again. What happens to the number of people, the number of new cases, and the AUC? Why does the number of people **fall**?
2. **Which predictor matters most?** Fit model 3 without `bmi` (`fit_logistic(setdiff(PREDICTORS, "bmi"))`), and compare its test AUC with model 3's.
3. **Does it work as well for everyone?** Compute the test AUC separately for women and men:

   ```r
   test_pred %>% group_by(sex_at_birth) %>% roc_auc(truth = t2d_future, .pred_yes, event_level = "second")
   ```

   We'll do this properly, for more groups, in **7.4**.

## 💬 Discuss

1. Which predictor would you **not** want in a model used in a clinic? (Think about income: is it fair to use it? Is it fair to leave it out?)
2. Our outcome is a **diagnosis**, not diabetes itself. People who see doctors more often are diagnosed sooner. How might that affect who the model calls "high risk"?
3. Would you trust this model in a hospital that isn't part of *All of Us*? What would you check first?

## Save for 7.2

7.2 uses the same people, outcome, and predictors, with tree-based models.

In [ ]:
save_data(ml_data, str_glue("ml_t2d_{HORIZON_YEARS}yr.csv"))

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *there is no package called 'rsample'* (or another) | the install didn't finish | rerun the setup cell; installing takes up to 15 minutes the first time |
| *Not run: over the limit* | the enrollment query costs more than `max_dollars` | check the printed estimate; raise the limit only knowingly |
| *747029 rows were requested, but only 584005 rows were received* | the results' rows are long (survey answer text), and the download asked for too many per page | fixed in `aou_setup.R` (November 2026): it downloads again in smaller pages, without running the query again. Seeing it? `git pull`, then rerun the setup cell |
| `Expected one education question` | the concept code lookup found 0 or several concepts | look at the `concepts` table; pick the question's ID by hand |
| An AUC above **0.95** | almost certainly **leakage** | check every predictor's date: was it known at enrollment? |
| An AUC **below 0.5** | the event level is the wrong way round | use `event_level = "second"`: the levels are `no`, `yes` |
| *glm.fit: fitted probabilities numerically 0 or 1 occurred* | some people are perfectly predicted | usually harmless here; with a tiny group, combine groups |
| Income groups show as `\n100k` or run together | JupyterLab reads text between two `$` signs as a maths formula | the labels have no `$` (`50k to 100k`); avoid `$` in group names |
| *Only first 100 results retrieved* after `last_query_cost()` | an old copy of `aou_setup.R` | harmless; `git pull` and rerun the setup cell to silence it |
| `tune_grid()` is slow | 5 folds × 20 penalties on many people | it's normal for it to take a few minutes |

## 🧑‍🏫 Instructor notes

- **Draw the timeline** first: time zero, predictors before it, the 2-year window after it. Most of the class follows from it.
- **"77% of cases were diagnosed before enrollment"** is the opening surprise: we can't predict what's already happened.
- **Ask "is this leakage?"** about a list: HbA1c, metformin, the number of doctor visits in the next year, a survey answered after diagnosis, BMI at enrollment. Only the last one is fine.
- **AUC vs calibration:** a model can rank well and still give wrong numbers. Ask: which matters for a patient told "your risk is 8%"?
- **LASSO ≈ logistic here** is a result, not a failure: more complex isn't better when the signal is simple. 7.2 tests that with trees.

## Key takeaways

- A **prediction** question has four parts: who, from what (known **when**?), predicting what, and **by when**.
- **Leakage** is avoided by dating every predictor, and starting the clock at **time zero**.
- Judge a model on a **test set** it never saw: **AUC** for ranking, **calibration** for the numbers.
- **Cross-validation** chooses settings on the training set, so the test set stays honest.
- A simple model with the right predictors is hard to beat.

## Clean up

**Clear outputs** before committing or sharing: **Edit → Clear Outputs of All Cells**, then save.

**Next:** **7.2 trees_and_boosting**: random forests and gradient boosting on the same data. Do flexible models find anything logistic regression missed?